In [3]:
import numpy as np
import cvxpy as cp
import scipy.linalg as la

import matplotlib.pyplot as plt

In [5]:
lamb = np.linspace(0,1,50)
theta = np.linspace(0,1,50)

In [9]:
def get_thermal_prob(epsilon, beta):
    """Calculate probability of |0> and |1> for a thermal state."""
    Z = 1 + np.exp(-beta * epsilon)
    p1 = np.exp(-beta * epsilon) / Z
    p0 = 1 - p1
    return p0, p1

def theorem_1_optimization(lamb,thet):
    print("--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---")
    
    # 1. 파라미터 설정
    epsilon = 1.0
    beta_A = 1.0  # Cold
    beta_B = 2  # Hot (beta가 작을수록 뜨거움)
    
    # 초기 상관관계 파라미터 (논문 Fig 2와 유사한 조건)
    # theta=0이면 Theorem 1에서는 AEF가 불가능해야 함 (Delta E_A >= 0)
    lam = lamb     # lambda
    theta = thet   # theta (이 값이 0보다 커야 퇴화 부분공간에 Coherence가 생김)

    # 2. 기저 정의 (Standard Basis: |00>, |01>, |10>, |11>)
    # Indices: 0->|00>, 1->|01>, 2->|10>, 3->|11>
    # Energies: 0, epsilon, epsilon, 2*epsilon
    
    # 3. 초기 상태 rho_AB 생성 (식 9)
    # 3-1. Thermal part
    pA_0, pA_1 = get_thermal_prob(epsilon, beta_A)
    pB_0, pB_1 = get_thermal_prob(epsilon, beta_B)
    
    # gamma_A tensor gamma_B (Diagonal matrix)
    gamma_AB_diag = [pA_0*pB_0, pA_0*pB_1, pA_1*pB_0, pA_1*pB_1]
    rho_thermal = np.diag(gamma_AB_diag)
    
    # 3-2. Bell States
    # |phi+> = (|00> + |11>) / sqrt(2) -> Impacts indices 0 and 3
    # |psi-> = (|01> - |10>) / sqrt(2) -> Impacts indices 1 and 2 (Degenerate subspace!)
    phi_plus = np.zeros((4, 4))
    phi_plus[0,0] = 0.5; phi_plus[3,3] = 0.5; phi_plus[0,3] = 0.5; phi_plus[3,0] = 0.5
    
    psi_minus = np.zeros((4, 4))
    psi_minus[1,1] = 0.5; psi_minus[2,2] = 0.5; psi_minus[1,2] = -0.5; psi_minus[2,1] = -0.5
    
    # Full rho_AB
    rho_AB = (1 - lam - theta) * rho_thermal + lam * phi_plus + theta * psi_minus

    # 4. 초기 A의 에너지 계산
    # A의 Hamiltonian in 4x4: H_A tensor I_B = diag(0, 0, epsilon, epsilon)
    H_A_4x4 = np.diag([0, 0, epsilon, epsilon])
    E_A_initial = np.real(np.trace(H_A_4x4 @ rho_AB))
    
    print(f"Initial Energy E_A: {E_A_initial:.4f}")

    # 5. Theorem 1 적용 (Block Optimization)
    # 에너지 보존 법칙에 의해 Unitary는 블록 대각 형태여야 함.
    # Block 1 (|00>): 변화 없음
    # Block 2 (|01>, |10>): 최적화 수행 (여기가 핵심)
    # Block 3 (|11>): 변화 없음
    
    # 5-1. Degenerate Subspace (|01>, |10>) 추출
    # Index 1, 2에 해당
    block_indices = [1, 2]
    rho_block = rho_AB[np.ix_(block_indices, block_indices)]
    
    # 5-2. 블록 대각화 (Eigenvalues 구하기)
    # 이 과정이 "Unitary를 통해 상관관계를 Population으로 변환"하는 과정임
    evals = la.eigvalsh(rho_block)
    # eigvalsh는 오름차순 정렬됨 (small, large)
    p_min, p_max = evals[0], evals[1]
    
    # 5-3. 에너지 최소화를 위한 재배치 (Passive State Logic)
    # A의 에너지를 낮추려면?
    # |01> (A=0, B=1) -> A 에너지 0
    # |10> (A=1, B=0) -> A 에너지 epsilon
    # 따라서 확률이 큰 값(p_max)을 |01>에 할당해야 함.
    
    # 최적화된 블록의 대각 성분 (Coherence는 제거됨)
    opt_block_diag = np.array([p_max, p_min]) 
    
    # 6. 최종 상태 구성 (Population만 업데이트)
    # Theorem 1에 따르면 최적 상태 sigma_A*는 대각 상태가 됨 (in Energy basis)
    rho_opt_diag = np.diag(rho_AB).copy()
    rho_opt_diag[1] = p_max  # |01>에 큰 확률
    rho_opt_diag[2] = p_min  # |10>에 작은 확률
    # |00>과 |11>은 건드리지 않음 (rho_opt_diag[0], [3] 유지)
    
    rho_opt = np.diag(rho_opt_diag)
    
    # 7. 최적화된 에너지 및 AEF 계산
    E_A_final = np.real(np.trace(H_A_4x4 @ rho_opt))
    Delta_E_A = E_A_final - E_A_initial
    
    print(f"Final Energy E_A:   {E_A_final:.4f}")
    print(f"Delta E_A (Thm 1):  {Delta_E_A:.4f}")
    
    if Delta_E_A < 0:
        print("=> AEF Activated! (Cooling achieved)")
    else:
        print("=> No AEF (Energy flowed naturally or no change)")

    return Delta_E_A

In [11]:
dataset = np.zeros((len(theta),len(lamb)))

for i in range(len(theta)):
    for j in range(len(lamb)):
        dataset[i][j] = theorem_1_optimization(i,j)

--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 0.2689
Final Energy E_A:   0.1192
Delta E_A (Thm 1):  -0.1497
=> AEF Activated! (Cooling achieved)
--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 0.5000
Final Energy E_A:   0.0000
Delta E_A (Thm 1):  -0.5000
=> AEF Activated! (Cooling achieved)
--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 0.7311
Final Energy E_A:   -0.1969
Delta E_A (Thm 1):  -0.9279
=> AEF Activated! (Cooling achieved)
--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 0.9621
Final Energy E_A:   -0.3956
Delta E_A (Thm 1):  -1.3577
=> AEF Activated! (Cooling achieved)
--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 1.1932
Final Energy E_A:   -0.5948
Delta E_A (Thm 1):  -1.7880
=> AEF Activated! (Cooling achieved)
--- [Gemini] Theorem 1 Simulation (Optimal Unitary) ---
Initial Energy E_A: 1.4242
Final Energy E_A:   -0.7942
Delta E_A (

In [12]:
dataset

array([[ -0.1497385 ,  -0.5       ,  -0.92792954, ..., -20.3070363 ,
        -20.73773898, -21.16844175],
       [  0.        ,  -0.43070508,  -0.86141016, ..., -20.24313884,
        -20.67384392, -21.104549  ],
       [  0.        ,  -0.37220174,  -0.80030622, ..., -20.1794721 ,
        -20.61017478, -21.04087756],
       ...,
       [  0.        ,  -0.03534541,  -0.13653768, ..., -17.54697708,
        -17.97287538, -18.3989633 ],
       [  0.        ,  -0.03461614,  -0.13385023, ..., -17.49348199,
        -17.91917663, -18.34506872],
       [  0.        ,  -0.03391623,  -0.13126478, ..., -17.4401966 ,
        -17.86568374, -18.29137628]], shape=(50, 50))

In [13]:
np.savetxt('./optdata.txt',dataset)